In [11]:
import pandas as pd
import glob

# Get all CSV files in the current directory
csv_files = glob.glob("*.csv")

# Map filenames to numerical identifiers
file_mapping = {file: i+1 for i, file in enumerate(csv_files)}

# Initialize an empty list to store DataFrames
df_list = []

# Process each CSV file
for file, identifier in file_mapping.items():
    # Read the file into a DataFrame
    df = pd.read_csv(file)
    
    # Replace seq_num with the identifier for source traceability
    df['seq_num'] = identifier
    
    # Append to the list
    df_list.append(df)

# Combine all DataFrames
combined_df = pd.concat(df_list, ignore_index=True)

# Sort by the rank column in ascending order
combined_df = combined_df.sort_values(by='rank', ascending=True)

# Save the combined DataFrame to a new CSV file
combined_df.to_csv("combined_epitopes.csv", index=False)

print("Combined CSV created successfully with ascending rank!")


Combined CSV created successfully with ascending rank!


In [21]:
import pandas as pd
from difflib import SequenceMatcher

# Function to check if two epitopes are similar above a given threshold
def is_similar(epitope1, epitope2, threshold=0.7):
    return SequenceMatcher(None, epitope1, epitope2).ratio() >= threshold

# Function to get unique epitopes from a DataFrame
def get_unique_epitopes(df, epitope_column, threshold, max_count):
    unique_epitopes = []
    rows_to_keep = []

    for index, row in df.iterrows():
        epitope = row[epitope_column]
        if all(not is_similar(epitope, unique, threshold) for unique in unique_epitopes):
            unique_epitopes.append(epitope)
            rows_to_keep.append(index)
        if len(unique_epitopes) == max_count:
            break

    return df.loc[rows_to_keep]

# Load the MHC-I and MHC-II files
mhc_i_file = "MHC_I_Epitopes.csv"
mhc_ii_file = "MHC_II_Epitopes.csv"

mhc_i_df = pd.read_csv(mhc_i_file)
mhc_ii_df = pd.read_csv(mhc_ii_file)

# Get the first 200 unique epitopes for each file
unique_mhc_i_df = get_unique_epitopes(mhc_i_df, "peptide", threshold=0.7, max_count=200)
unique_mhc_ii_df = get_unique_epitopes(mhc_ii_df, "peptide", threshold=0.7, max_count=200)

# Save the results to new CSV files
unique_mhc_i_df.to_csv("Unique_MHC_I_Epitopes.csv", index=False)
unique_mhc_ii_df.to_csv("Unique_MHC_II_Epitopes.csv", index=False)

print("Unique epitopes saved to 'Unique_MHC_I_Epitopes.csv' and 'Unique_MHC_II_Epitopes.csv'")


Unique epitopes saved to 'Unique_MHC_I_Epitopes.csv' and 'Unique_MHC_II_Epitopes.csv'
